# Mahalanobis Distance for Multivariate Outlier Detection

**Course:** 25MSDA3H01 – Multivariate Data Analysis  
**Objective:** Calculate Mahalanobis distance, identify multivariate outliers, visualize feature relationships, and validate the implementation against SciPy.

This notebook is reproducible. It uses a fixed random seed and includes a CSV dataset in `data/`.


## 1. Import libraries and load the dataset

NumPy is used for matrix operations, Pandas for data handling, Matplotlib/Seaborn for visualization, and SciPy for independent validation.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

DATA_PATH = Path("data/multivariate_dataset.csv")
df = pd.read_csv(DATA_PATH)
display(df.head())
print("Dataset shape:", df.shape)
print("\\nMissing values:\\n", df.isnull().sum())


## 2. Select numerical variables

The ID and known demonstration label are not used in the distance calculation. The label is only used after detection to evaluate the injected examples.


In [ ]:
features = ["Feature_1", "Feature_2", "Feature_3"]
X = df[features].to_numpy(dtype=float)
n, p = X.shape
print(f"Observations: {n}, variables: {p}")


## 3. Calculate mean vector and covariance matrix from scratch

For a data matrix X, the sample covariance matrix is:

\[
S = \\frac{1}{n-1}(X-\\bar X)^T(X-\\bar X)
\]

The denominator is `n - 1` because this is the sample covariance.


In [ ]:
mean_vector = np.mean(X, axis=0)
centered = X - mean_vector
covariance_manual = (centered.T @ centered) / (n - 1)

print("Mean vector:")
print(mean_vector)
print("\\nCovariance matrix calculated manually:")
print(covariance_manual)


## 4. Calculate Mahalanobis distance

\[
D_M(x)=\\sqrt{(x-\\mu)^T S^{-1}(x-\\mu)}
\]

We calculate the squared distance for each row using NumPy matrix operations. A pseudo-inverse is a possible fallback if the covariance matrix is singular; here we use the ordinary inverse and report an error if the variables are linearly dependent.


In [ ]:
covariance_inverse = np.linalg.inv(covariance_manual)
difference = X - mean_vector
squared_distance = np.einsum(
    "ij,jk,ik->i", difference, covariance_inverse, difference
)
mahalanobis_distance = np.sqrt(np.maximum(squared_distance, 0))

results = df[["Record_ID", "Known_Label"]].copy()
results["Mahalanobis_Distance"] = mahalanobis_distance
results["Squared_Mahalanobis_Distance"] = squared_distance
display(results.head())


## 5. Detect multivariate outliers

Under the multivariate normality assumption, squared Mahalanobis distances can be compared with a chi-square threshold with `p` degrees of freedom. This notebook uses the 99th percentile. This is a screening rule; real datasets should also be checked for distributional assumptions and influential observations.


In [ ]:
from scipy.stats import chi2

confidence_level = 0.99
cutoff = chi2.ppf(confidence_level, df=p)
results["Outlier_Detected"] = results["Squared_Mahalanobis_Distance"] > cutoff

print(f"Chi-square cutoff ({confidence_level:.0%}, df={p}): {cutoff:.4f}")
print("Detected outliers:", int(results["Outlier_Detected"].sum()))
display(results[results["Outlier_Detected"]].sort_values(
    "Mahalanobis_Distance", ascending=False
))


## 6. Validate against standard library output

SciPy's `mahalanobis` function accepts the inverse covariance matrix. We compare its distances with our NumPy implementation. The maximum absolute difference should be close to zero (allowing for floating-point rounding).


In [ ]:
from scipy.spatial.distance import mahalanobis

scipy_distances = np.array([
    mahalanobis(row, mean_vector, covariance_inverse) for row in X
])
max_abs_difference = np.max(np.abs(mahalanobis_distance - scipy_distances))
print("Maximum absolute difference:", max_abs_difference)
assert np.allclose(mahalanobis_distance, scipy_distances, atol=1e-8)
print("Validation passed: manual NumPy results match SciPy.")


## 7. Matrix scatter plot

A pair plot shows each variable against every other variable. Points marked as detected outliers are highlighted separately.


In [ ]:
plot_data = df[features].copy()
plot_data["Detection"] = np.where(
    results["Outlier_Detected"], "Detected outlier", "Not detected"
)
sns.pairplot(
    plot_data, vars=features, hue="Detection",
    diag_kind="hist", plot_kws={"alpha": 0.7, "s": 25}
)
plt.suptitle("Pairwise Scatter Matrix - Mahalanobis Outlier Detection", y=1.02)
plt.show()


## 8. Save calculated outputs

The script writes the distance table, covariance matrix, and a text summary to the `outputs/` folder. These files are included in the repository as example outputs.


In [ ]:
from pathlib import Path

output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

results.to_csv(output_dir / "mahalanobis_results.csv", index=False)
pd.DataFrame(
    covariance_manual, index=features, columns=features
).to_csv(output_dir / "covariance_matrix.csv")

summary = {
    "observations": n,
    "variables": p,
    "confidence_level": confidence_level,
    "chi_square_cutoff": cutoff,
    "detected_outliers": int(results["Outlier_Detected"].sum()),
    "max_abs_difference_vs_scipy": float(max_abs_difference)
}
pd.Series(summary, name="value").to_csv(output_dir / "metrics_summary.csv")
print("Outputs saved in:", output_dir.resolve())


## 9. Interpretation and conclusion

- Mahalanobis distance measures how far a row is from the multivariate mean while accounting for correlations and different variable scales.
- A large squared distance relative to the chi-square cutoff indicates a potential multivariate outlier.
- The pair plot helps visually inspect relationships among variables and flagged observations.
- The SciPy comparison checks that the manual NumPy calculation is implemented correctly.

**Important:** The sample dataset contains five deliberately injected unusual observations for demonstration. On a real dataset, do not use a known label to calculate distances; use labels only for evaluation if they are available.
